# 02.4 - Graph Algorithms

Builds on `01-Data Structures/01.7-Graphs` (representation, BFS, DFS).

| Algorithm | Solves | Time | Real use |
|---|---|---|---|
| **BFS** | Shortest path (unweighted) | O(V + E) | Social "degrees", web crawlers, puzzles |
| **DFS** | Connectivity, cycles, paths | O(V + E) | Maze solving, dependency checks |
| **Topological sort** | Order tasks with dependencies (DAG) | O(V + E) | pip/npm installs, build systems (make), course prerequisites, Excel recalculation, Airflow pipelines |
| **Dijkstra** | Shortest path, non-negative weights | O((V + E) log V) | **Google Maps, network routing (OSPF)**, ride-hailing ETA |
| **Bellman-Ford** | Shortest path with negative weights | O(V · E) | Currency arbitrage detection, RIP routing |
| **Kruskal / Prim (MST)** | Cheapest way to connect everything | O(E log E) | Laying cables, roads, power grids, clustering |

In [1]:
from collections import defaultdict, deque
import heapq

## 1. BFS shortest path on a grid
Shortest path in a maze (each move costs 1) → BFS. Games, robots and warehouse pathfinding use this (A* adds a heuristic on top).

In [2]:
def shortest_path_grid(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    q = deque([(start, 0)])
    seen = {start}
    while q:
        (r, c), dist = q.popleft()
        if (r, c) == goal:
            return dist
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "." and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append(((nr, nc), dist + 1))
    return -1

maze = ["..#....",
        ".##.##.",
        "....#..",
        "#.#...#",
        "...#..."]
print("steps:", shortest_path_grid(maze, (0, 0), (4, 6)))

steps: 10


## 2. DFS: cycle detection in a directed graph (3 colours)
- **white (0)** = unvisited, **gray (1)** = on the current DFS path, **black (2)** = finished.
- Reaching a **gray** node again = a back edge = **cycle** (e.g. circular imports, deadlocks).

In [3]:
def has_cycle_directed(n, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
    state = [0] * n
    def dfs(u):
        state[u] = 1
        for v in adj[u]:
            if state[v] == 1:
                return True
            if state[v] == 0 and dfs(v):
                return True
        state[u] = 2
        return False
    return any(state[u] == 0 and dfs(u) for u in range(n))

print(has_cycle_directed(3, [(0, 1), (1, 2)]), has_cycle_directed(3, [(0, 1), (1, 2), (2, 0)]))

False True


## 3. Topological sort
Order nodes so every edge `u → v` has `u` before `v`. Only possible in a **DAG** (no cycles).
- **Kahn's algorithm (BFS):** repeatedly take nodes with in-degree 0.
- **DFS:** add a node after all its descendants finish, then reverse.

In [4]:
def topo_sort_kahn(graph):
    indegree = defaultdict(int)
    for u in graph:
        for v in graph[u]:
            indegree[v] += 1
    nodes = set(graph) | set(indegree)
    q = deque(sorted(n for n in nodes if indegree[n] == 0))
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph.get(u, []):
            indegree[v] -= 1
            if indegree[v] == 0:
                q.append(v)
    return order if len(order) == len(nodes) else None    # None -> cycle

# Real-world: install order of Python packages (package -> packages that depend on it)
deps = {
    "numpy": ["pandas", "scipy", "matplotlib"],
    "pandas": ["seaborn"],
    "matplotlib": ["seaborn"],
    "scipy": ["scikit-learn"],
    "seaborn": [],
    "scikit-learn": [],
}
print("install order:", topo_sort_kahn(deps))

install order: ['numpy', 'pandas', 'scipy', 'matplotlib', 'scikit-learn', 'seaborn']


## 4. Dijkstra's shortest path For weighted graphs with **non-negative** weights. Greedy + heap:
1. Distance to start = 0, everything else = ∞.
2. Pop the closest unfinished node from a min-heap.
3. **Relax** its edges: if `dist[u] + w < dist[v]`, update and push `v`.

This is the core of Google Maps / GPS routing (with many optimisations on top).

In [5]:
def dijkstra(graph, source):
    dist = {source: 0}
    prev = {}
    heap = [(0, source)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist.get(u, float("inf")):
            continue                              # stale entry
        for v, w in graph.get(u, []):
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(heap, (nd, v))
    return dist, prev

def path_to(prev, target):
    path = [target]
    while path[-1] in prev:
        path.append(prev[path[-1]])
    return path[::-1]

# Cities with illustrative (made-up) travel times in minutes
roads = {
    "Kigali":   [("Muhanga", 60), ("Musanze", 100), ("Rwamagana", 70)],
    "Muhanga":  [("Huye", 75), ("Karongi", 90)],
    "Musanze":  [("Rubavu", 70)],
    "Karongi":  [("Rubavu", 120), ("Rusizi", 180)],
    "Huye":     [("Rusizi", 150)],
    "Rwamagana": [],
    "Rubavu":   [],
    "Rusizi":   [],
}
dist, prev = dijkstra(roads, "Kigali")
for city in ["Huye", "Rubavu", "Rusizi"]:
    print(f"{city:8} {dist[city]:4} min via {' -> '.join(path_to(prev, city))}")

Huye      135 min via Kigali -> Muhanga -> Huye
Rubavu    170 min via Kigali -> Musanze -> Rubavu
Rusizi    285 min via Kigali -> Muhanga -> Huye -> Rusizi


## 5. Bellman-Ford (handles negative weights, detects negative cycles)
Relax **every edge** V−1 times. If anything still improves after that, there's a negative cycle.
Real use: in currency exchange, a negative cycle in `-log(rate)` = an **arbitrage** opportunity.

In [6]:
def bellman_ford(n, edges, source):
    dist = [float("inf")] * n
    dist[source] = 0
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            return None                       # negative cycle
    return dist

print(bellman_ford(4, [(0, 1, 4), (0, 2, 5), (1, 2, -3), (2, 3, 4)], 0))
print(bellman_ford(3, [(0, 1, 1), (1, 2, -1), (2, 1, -1)], 0))

[0, 4, 1, 5]
None


## 6. Minimum Spanning Tree (MST)
Connect **all** nodes with the **minimum total edge cost** (no cycles).
- **Kruskal:** sort edges by cost, add an edge if it doesn't create a cycle (Union-Find, see 01.9).
- **Prim:** grow from one node, always add the cheapest edge leaving the tree (heap).

In [7]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.parent[rb] = ra
        return True

def kruskal(n, edges):
    uf, total, used = UnionFind(n), 0, []
    for w, u, v in sorted(edges):
        if uf.union(u, v):
            total += w
            used.append((u, v, w))
    return total, used

def prim(n, adj):
    seen, total, heap = set(), 0, [(0, 0)]
    while heap and len(seen) < n:
        w, u = heapq.heappop(heap)
        if u in seen:
            continue
        seen.add(u)
        total += w
        for v, wv in adj[u]:
            if v not in seen:
                heapq.heappush(heap, (wv, v))
    return total

# Laying fibre cable between 5 offices (cost in $1000s)
edges = [(4, 0, 1), (8, 0, 2), (2, 1, 2), (6, 1, 3), (3, 2, 3), (9, 3, 4), (5, 2, 4)]
total, used = kruskal(5, edges)
adj = defaultdict(list)
for w, u, v in edges:
    adj[u].append((v, w)); adj[v].append((u, w))
print("Kruskal:", total, used)
print("Prim   :", prim(5, adj))
assert total == prim(5, adj) == 14

Kruskal: 14 [(1, 2, 2), (2, 3, 3), (0, 1, 4), (2, 4, 5)]
Prim   : 14


---
## Solved problems

### 1. Network Delay Time - LC #743 (Medium) Dijkstra
Signal sent from node `k`; `times[i] = (u, v, w)`. Time until all `n` nodes receive it, or -1.

In [8]:
def network_delay_time(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {}
    heap = [(0, k)]
    while heap:
        d, u = heapq.heappop(heap)
        if u in dist:
            continue
        dist[u] = d                          # first pop = shortest distance
        for v, w in graph[u]:
            if v not in dist:
                heapq.heappush(heap, (d + w, v))
    return max(dist.values()) if len(dist) == n else -1

assert network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay_time([[1, 2, 1]], 2, 1) == 1
assert network_delay_time([[1, 2, 1]], 2, 2) == -1
print("Network Delay ")

Network Delay 


### 2. Course Schedule II - LC #210 (Medium) topological sort
Return an order to take all courses (`[course, prereq]` pairs), or `[]` if impossible.

In [9]:
def find_order(num_courses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * num_courses
    for course, pre in prerequisites:
        graph[pre].append(course)
        indegree[course] += 1
    q = deque(i for i in range(num_courses) if indegree[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                q.append(v)
    return order if len(order) == num_courses else []

assert find_order(2, [[1, 0]]) == [0, 1]
order = find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
assert order[0] == 0 and order[-1] == 3
assert find_order(2, [[0, 1], [1, 0]]) == []
print("Course Schedule II ")

Course Schedule II 


### 3. Word Ladder - LC #127 (Hard) BFS
Shortest transformation `beginWord → endWord`, changing one letter at a time, each step a dictionary word.

**Idea:** Words are nodes; one-letter differences are edges; shortest path in an unweighted graph → BFS. Wildcard patterns find neighbours fast.

In [10]:
def ladder_length(begin_word, end_word, word_list):
    words = set(word_list)
    if end_word not in words:
        return 0
    # pattern "h*t" -> ["hot", "hit", ...]  (avoids comparing every pair of words)
    patterns = defaultdict(list)
    for w in words | {begin_word}:
        for i in range(len(w)):
            patterns[w[:i] + "*" + w[i + 1:]].append(w)
    q = deque([(begin_word, 1)])
    seen = {begin_word}
    while q:
        word, steps = q.popleft()
        if word == end_word:
            return steps
        for i in range(len(word)):
            for nxt in patterns[word[:i] + "*" + word[i + 1:]]:
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, steps + 1))
    return 0

assert ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]) == 5
assert ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"]) == 0
print("Word Ladder ")

Word Ladder 


### 4. Min Cost to Connect All Points - LC #1584 (Medium) MST
Connect all points; cost of an edge = Manhattan distance. Minimum total cost.

**Idea:** Prim's algorithm on the complete graph of points.

In [11]:
def min_cost_connect_points(points):
    n = len(points)
    seen = set()
    heap = [(0, 0)]
    total = 0
    while len(seen) < n:
        cost, i = heapq.heappop(heap)
        if i in seen:
            continue
        seen.add(i)
        total += cost
        xi, yi = points[i]
        for j in range(n):
            if j not in seen:
                xj, yj = points[j]
                heapq.heappush(heap, (abs(xi - xj) + abs(yi - yj), j))
    return total

assert min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]) == 20
assert min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]) == 18
print("Min Cost Connect Points ")

Min Cost Connect Points 


### 5. Cheapest Flights Within K Stops - LC #787 (Medium) - Bellman-Ford variant
Cheapest price from `src` to `dst` using at most `k` stops.

**Idea:** Bellman-Ford limited to k+1 rounds of relaxation; copy the array each round so a round adds at most one flight.

In [12]:
def find_cheapest_price(n, flights, src, dst, k):
    prices = [float("inf")] * n
    prices[src] = 0
    for _ in range(k + 1):                   # k stops = k + 1 flights
        new_prices = prices[:]               # copy: only use last round's values
        for u, v, w in flights:
            if prices[u] + w < new_prices[v]:
                new_prices[v] = prices[u] + w
        prices = new_prices
    return -1 if prices[dst] == float("inf") else prices[dst]

flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
assert find_cheapest_price(4, flights, 0, 3, 1) == 700
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1) == 200
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0) == 500
print("Cheapest Flights ")

Cheapest Flights 


## More practice on LeetCode
Medium: Shortest Path in Binary Matrix (#1091), Path with Minimum Effort (#1631), Graph Valid Tree (#261), Evaluate Division (#399)
Hard: Alien Dictionary (#269 - topological sort), Reconstruct Itinerary (#332), Swim in Rising Water (#778)

## Key takeaways
| Question says… | Use |
|---|---|
| shortest path, all edges equal | **BFS** |
| shortest path, positive weights | **Dijkstra** (heap) |
| negative weights / at most k edges | **Bellman-Ford** |
| order with dependencies | **Topological sort** |
| connect everything cheaply | **MST** (Kruskal + Union-Find, or Prim + heap) |
| cycle in directed graph | DFS with 3 colours (or Kahn: not all nodes processed) |

**Next: `02.5-Greedy-Algorithms.ipynb`**